In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

# 커스텀 팔레트
custom_cmap_logo = sns.blend_palette(['#10181e', '#e5e8e9', '#2ab1dc'], as_cmap=True) # 핏큘레이터 로고에서 따왔습니다
custom_cmap_rextreme = sns.blend_palette(['#c5d943', '#2a3457', '#0dabbe'], as_cmap=True) # 이건 렉스트림에서 따왔습니다

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap_logo)
plt.show()

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap_rextreme)
plt.show()

# 그래프 기본 테마 설정
sns.set_theme(style="whitegrid", font_scale=1) # 블루톤

# 그리드 색상 조절
plt.rcParams.update({
    "grid.color": ".8",          # 그리드 색상
    "grid.linestyle": "--",       # 그리드 점선 스타일
    "grid.linewidth": 0.8,        # 그리드 두께
    "axes.grid": True,            # 그리드 항상 켜기
    "axes.edgecolor": ".8",       # 축 테두리 색상
})

# 막대그래프 관련 설정
plt.rcParams.update({
    "lines.linewidth": 2,
    "lines.marker": "D",          # 전역 마커 설정
    "lines.markersize": 7        # 마커 크기
})

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔스퀘어
plt.rcParams['mathtext.fontset'] = 'dejavusans'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 15, 9
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'cm'

In [ ]:
# 알아서 째라...
def get_palette(n, cmap):
    return [cmap(i / (n - 1)) for i in range(n)]

# 데이터 열기 & 전처리

In [ ]:
def split_ga4_csv(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    tables = []
    current_table = []

    for line in lines:
        # 줄바꿈만 있거나 쉼표만 있는 빈 줄을 만나면 지금까지의 데이터를 저장
        if line.strip() == "" or line.strip().replace(',', '') == "":
            if current_table:
                # 메모리 상에서 해당 부분만 데이터프레임으로 변환
                df = pd.read_csv(io.StringIO("".join(current_table)), comment='#')
                tables.append(df)
                current_table = []
        else:
            current_table.append(line)

    # 마지막 테이블 처리
    if current_table:
        tables.append(pd.read_csv(io.StringIO("".join(current_table))))

    return tables

# 사용 예시
all_tables = split_ga4_csv('data/20260319_Rextreme_GA4/Generate_leads_overview.csv')

# GA4 overview 파일의 경우 보통:
# all_tables[0] -> 상단 요약 정보 (2컬럼)
# all_tables[1] -> 중간 상세 내역 (N컬럼)

In [ ]:
event_df = pd.read_csv('data/20260319_Rextreme_GA4/Events_Event_name.csv', comment='#')

event_df

## .info()

In [ ]:
event_df.info()

## .describe()

In [ ]:
event_df.describe()

In [ ]:
event_df.describe(include='O')

## .isna().sum()

In [ ]:
event_df.isna().sum()

In [ ]:
not_set_counts = (event_df == '(not set)').sum()

not_set_counts

## .head()

In [ ]:
event_df.head()

## .columns

In [ ]:
event_df.columns

## .shape

In [ ]:
event_df.shape

# 얘 전처리 필요 없어보이는데 걍 가죠?

## Event count 많은 순

In [ ]:
n = event_df.shape[0]
cmap = custom_cmap_logo

ax = sns.barplot(event_df, x = 'Event name', y = 'Event count', hue = 'Event name', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('Event count 많은 순')
plt.xlabel('국가')
plt.ylabel('Event count')
plt.xticks(rotation=90)

plt.show()

### 초기 이탈률

In [ ]:
# 값만 추출해서 계산 (iloc[0]이나 values[0] 사용)
s_count = event_df[event_df['Event name'] == 'session_start']['Event count'].values[0]
sc_count = event_df[event_df['Event name'] == 'scroll']['Event count'].values[0]

diff = s_count - sc_count
drop_rate = (diff / s_count) * 100

print(f"--- 분석 결과 ---")
print(f"세션 시작: {s_count:,}건")
print(f"스크롤 발생: {sc_count:,}건")
print(f"차이(이탈 예상): {diff:,}건")
print(f"초기 이탈률: {drop_rate:.2f}%")

### 재방문율

In [ ]:
# 값 추출
total_sessions = event_df[event_df['Event name'] == 'session_start']['Event count'].values[0]
new_users = event_df[event_df['Event name'] == 'first_visit']['Event count'].values[0]

# 재방문 세션 수 계산
returning_sessions = total_sessions - new_users
# 재방문 비중(%)
returning_rate = (returning_sessions / total_sessions) * 100

print(f"전체 방문 중 재방문 비중: {returning_rate:.2f}%")

In [ ]:
event_df_sort = event_df.sort_values('Total users', ascending = False)
event_df_sort

## Total user 많은 순

In [ ]:
n = event_df.shape[0]
cmap = custom_cmap_logo
event_df_sort = event_df.sort_values('Total users', ascending = False)

ax = sns.barplot(event_df_sort, x = 'Event name', y = 'Total users', hue = 'Event name', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('Total users 많은 순')
plt.xlabel('국가')
plt.ylabel('Total users')
plt.xticks(rotation=90)

plt.show()

### Event count per active user 많은 순

In [ ]:
n = event_df.shape[0]
cmap = custom_cmap_logo
event_df_sort = event_df.sort_values('Event count per active user', ascending = False)

ax = sns.barplot(event_df_sort, x = 'Event name', y = 'Event count per active user', hue = 'Event name', palette = get_palette(n, cmap))

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='edge',
                    padding=3, fontsize=10, fontweight='bold')

plt.title('Event count per active user 많은 순')
plt.xlabel('국가')
plt.ylabel('Event count per active user')
plt.xticks(rotation=90)

plt.show()